In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from MuyGPyS._test.sampler import print_results
from MuyGPyS.neighbors import NN_Wrapper
from MuyGPyS.gp import MuyGPS
from MuyGPyS.gp.deformation import Isotropy, l2
from MuyGPyS.gp.hyperparameter import AnalyticScale, Parameter
from MuyGPyS.gp.kernels import Matern
from MuyGPyS.gp.noise import HomoscedasticNoise
from MuyGPyS.optimize import Bayes_optimize, L_BFGS_B_optimize
from MuyGPyS.optimize.batch import sample_batch
from MuyGPyS.optimize.loss import lool_fn

# Import and setup data

In [2]:
aqi_data = np.genfromtxt("biggest_day_aqi_coords_scaled.csv", delimiter=',', skip_header=1)
print(aqi_data[:10])

[[ 0.27128141  0.75872801 53.        ]
 [ 0.33042753  0.78042228 47.        ]
 [ 0.36178224  0.76125781 69.        ]
 [ 0.35172346  0.7786855  69.        ]
 [ 0.31374196  0.77694373 48.        ]
 [ 0.2842055   0.76718129 52.        ]
 [ 0.34535027  0.77844838 65.        ]
 [ 0.33611686  0.76985651 74.        ]
 [ 0.35287642  0.76437478 63.        ]
 [ 0.36015016  0.76916901 45.        ]]


In [3]:
aqi_x, aqi_y = aqi_data[:, :-1], aqi_data[:, -1]
aqi_x_train, aqi_x_test, aqi_y_train, aqi_y_test = train_test_split(
    aqi_x, aqi_y, test_size=0.1, random_state=24
)

# FIX 1: Subtract the training mean (constant mean function µ1).
# GPs assume a zero-mean process. AQI values are in the range ~20-200, so
# failing to de-mean inflates the leave-one-out loss by ~mean^2 per point,
# which is why the optimization target never dropped below -6000.
# We'll add the mean back to our predictions at inference time.
aqi_y_mean = aqi_y_train.mean()
aqi_y_train_centered = aqi_y_train - aqi_y_mean
aqi_y_test_centered  = aqi_y_test  - aqi_y_mean

print(f"Training mean (µ1): {aqi_y_mean:.3f}")
print(f"Centered train responses — mean: {aqi_y_train_centered.mean():.4f},  std: {aqi_y_train_centered.std():.3f}")
print(f"Feature shape: {aqi_x_train.shape}")

Training mean (µ1): 46.126
Centered train responses — mean: -0.0000,  std: 16.165
Feature shape: (989, 2)


In [24]:
print(aqi_y_test_centered[:5])

[-18.12639029 -15.12639029  12.87360971  11.87360971  30.87360971]


# Nearest Neighbors and Batches

In [4]:
nn_count = 30
nbrs_lookup = NN_Wrapper(aqi_x_train, nn_count, nn_method="exact", algorithm="ball_tree")

In [5]:
batch_count = 1500
batch_indices, batch_nn_indices = sample_batch(
    nbrs_lookup, batch_count, len(aqi_x_train)
)

# Setting and Optimizing Hyperparameters

In [6]:
aqi_muygps = MuyGPS(
    kernel=Matern(
        smoothness=Parameter("log_sample", (0.1, 4.0)),
        deformation=Isotropy(
            l2,
            # FIX 2: Widen the length_scale upper bound.
            # The optimizer was repeatedly hitting 0.253-0.299 against the
            # old ceiling of 0.3, suggesting the true optimum is larger.
            length_scale=Parameter("log_sample", (0.01, 2.0)),
        ),
    ),
    # FIX 3: Widen the noise bounds substantially.
    # After de-meaning, residuals still have variance in the tens-of-AQI-units
    # range. The old upper bound of 1e-3 forced noise to near-zero, making
    # the model artificially overconfident and yielding poor uncertainty estimates.
    # After de-meaning, typical residual noise for AQI data might be 0.01–1.0,
    # so allow the optimizer to explore that range.
    noise=HomoscedasticNoise("sample", (1e-6, 1.0)),
    scale=AnalyticScale(),
)

In [12]:
# Build training tensors using the CENTERED responses.
(
    batch_crosswise_dists,
    batch_pairwise_dists,
    batch_ys,
    batch_nn_ys,
) = aqi_muygps.make_train_tensors(
    batch_indices,
    batch_nn_indices,
    aqi_x_train,
    aqi_y_train_centered,   # <-- centered responses
)

# Optimize hyperparameters against the leave-one-out log-likelihood.
aqi_muygps_optimized = Bayes_optimize(
    aqi_muygps,
    batch_ys,
    batch_nn_ys,
    batch_crosswise_dists,
    batch_pairwise_dists,
    loss_fn=lool_fn,
    verbose=True,
    random_state=24,
    init_points=15,
    n_iter=25,
)

parameters to be optimized: ['length_scale', 'smoothness', 'noise']
bounds: [[1.e-02 2.e+00]
 [1.e-01 4.e+00]
 [1.e-06 1.e+00]]
initial x0: [0.26597923 0.18871263 0.59450688]
|   iter    |  target   | length... | smooth... |   noise   |
-------------------------------------------------------------
| 1         | -7094.660 | 0.2659792 | 0.1887126 | 0.5945068 |
| 2         | -21230.62 | 1.9204344 | 2.8280969 | 0.9998672 |
| 3         | -25256.79 | 0.4479339 | 1.5081197 | 0.7398412 |
| 4         | -128006.6 | 1.9929468 | 1.3337532 | 0.1365454 |
| 5         | -47682.59 | 0.7741202 | 1.3500252 | 0.3664153 |
| 6         | -36227.74 | 1.4222066 | 3.6105554 | 0.5341159 |
| 7         | -33153.74 | 0.5021145 | 2.7200455 | 0.5617295 |
| 8         | -24251.39 | 1.0896941 | 3.5844456 | 0.8427797 |
| 9         | -28454.72 | 0.6189650 | 2.5615621 | 0.6802391 |
| 10        | -22299.38 | 1.9411508 | 3.5849118 | 0.9424259 |
| 11        | -79087.49 | 1.2880287 | 2.4971257 | 0.2276840 |
| 12        | -2415

In [13]:
# Analytically estimate the variance scale σ² from the centered neighbors.
aqi_muygps_optimized = aqi_muygps_optimized.optimize_scale(
    batch_pairwise_dists,
    batch_nn_ys
)

# Inference

In [14]:
test_count = aqi_x_test.shape[0]
test_indices = np.arange(test_count)
test_nn_indices, _ = nbrs_lookup.get_nns(aqi_x_test)

In [15]:
# Build prediction tensors using the CENTERED training responses.
(
    test_crosswise_dists,
    test_pairwise_dists,
    test_nn_ys,
) = aqi_muygps_optimized.make_predict_tensors(
    test_indices,
    test_nn_indices,
    aqi_x_test,
    aqi_x_train,
    aqi_y_train_centered,   # <-- centered responses
)

kcross = aqi_muygps_optimized.kernel(test_crosswise_dists)
kin    = aqi_muygps_optimized.kernel(test_pairwise_dists)

In [17]:
# Posterior mean is in the centered space; add the training mean back.
predictions_centered = aqi_muygps_optimized.posterior_mean(kin, kcross, test_nn_ys)
predictions = predictions_centered + aqi_y_mean   # FIX 4: un-center

variances = aqi_muygps_optimized.posterior_variance(kin, kcross)
confidence_intervals = np.sqrt(variances) * 1.96

# Evaluate against the ORIGINAL (un-centered) test responses.
coverage = np.count_nonzero(
    np.abs(aqi_y_test - predictions) < confidence_intervals
) / test_count

rmse = np.sqrt(np.mean((aqi_y_test - predictions) ** 2))
print(f"RMSE: {rmse:.4f}")
print(f"Coverage: {coverage:.4f}")

RMSE: 16.5216
Coverage: 0.8909


In [23]:
print(coverage)
print()
print(aqi_y_test[:5])
print()
print(predictions[:5])
print()
print(confidence_intervals[:5])
print()
print(test_count)


0.8909090909090909

[28. 31. 59. 58. 77.]

[38.03545698 40.06718016 53.03652689 50.677383   51.09276828]

[27.70663661 26.08471915 27.78027316 26.59992959 26.97656561]

110


In [18]:
# print_results expects un-centered responses and un-centered predictions.
print_results(
    aqi_y_test,
    ("optimized", aqi_muygps_optimized, predictions, variances, confidence_intervals, coverage)
)

name,smoothness,length scale,noise variance,variance scale,rmse,mean variance,mean confidence interval,coverage
optimized,0.100000,0.010000,0.000001,224.722651,16.521558,192.224908,27.149088,0.890909
